[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/intronlp-2026/blob/main/notebooks/week-08.ipynb)

# 8주차 실습: 관광 안내 검색기 만들고 문장 유사도 재기

**목표.** 관광 안내 문서 12개로 TF-IDF 검색기를 만들고, 같은 질문을 다르게 물었을 때 결과가 어떻게 갈리는지 본다. 글자 조각 TF-IDF와 문장 임베딩 모델로 고쳐 보고, 내가 만든 문장쌍으로 세 방법을 비교한다.

## 0. 준비

TF-IDF를 계산할 scikit-learn, 표를 보여 줄 pandas, 문장 임베딩 모델을 쓸 sentence-transformers를 설치합니다. Colab에는 대부분 이미 설치되어 있습니다.

설치 셀에서 오류가 나면 다음 셀로 넘어가지 말고 오류의 마지막 줄을 확인하세요.

In [ ]:
%pip -q install "scikit-learn>=1.3" "pandas>=1.5.3" "sentence-transformers>=3"

In [ ]:
import sys
import sklearn
import pandas
import sentence_transformers

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("pandas:", pandas.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("설치 확인 끝")

## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.

### 1-1. 관광 안내 문서 12개

제주 여행 안내 문장 12개를 검색할 문서로 씁니다. 문서는 **설명용 예시**입니다. 실제 운영 정보는 공식 안내를 확인하세요.

In [ ]:
docs = [
    "제주국제공항에서 제주시내까지는 리무진 버스나 시내버스를 타면 된다",
    "성산일출봉은 해가 뜨기 전에 올라가면 정상에서 일출을 볼 수 있다",
    "우도에 가려면 성산포항에서 배를 타고 들어가야 한다",
    "한라산 일부 탐방로는 미리 탐방 예약을 해야 오를 수 있다",
    "비가 오는 날에는 실내 관광지나 박물관을 둘러보기 좋다",
    "제주 올레길은 여러 코스로 나뉘어 해안을 따라 걸을 수 있다",
    "고기국수는 돼지고기 수육을 얹은 제주의 국수 요리다",
    "렌터카가 없어도 버스로 주요 관광지를 다닐 수 있다",
    "감귤 따기 체험은 주로 겨울철 농장에서 열린다",
    "협재 해수욕장은 물이 맑고 얕아 아이와 함께 가기 좋다",
    "해녀박물관에서는 제주 해녀의 물질 문화를 배울 수 있다",
    "제주시 숙소는 공항과 가까워 이동이 편하다",
]
for i, d in enumerate(docs, start=1):
    print(f"{i:2d}. {d}")

### 1-2. TF-IDF로 문서를 벡터로 바꾸기

5주차 단어 가방처럼 문서를 단어 칸의 숫자로 바꿉니다. 다만 칸의 숫자가 횟수가 아니라 **TF-IDF 점수**입니다.

- **TF**: 이 문서에 그 단어가 몇 번 나왔나. 많이 나올수록 크다
- **IDF**: 그 단어가 몇 개의 문서에 나오나. **여러 문서에 흔하게 나올수록 작다**

두 값을 곱하면, 이 문서에 자주 나오면서 다른 문서에는 드문 단어가 높은 점수를 받습니다. 아래에서 `수` 와 `있다` 처럼 여러 문서에 나오는 단어의 IDF가 낮은지, 그래서 2번 문서 안에서도 점수가 가장 낮은지 보세요.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 띄어쓰기 단위로 자르고, 한 글자 단어(배, 수)도 남긴다
word_tfidf = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
doc_vectors = word_tfidf.fit_transform(docs)
print("문서", doc_vectors.shape[0], "개 x 단어 칸", doc_vectors.shape[1], "개")

idf = dict(zip(word_tfidf.get_feature_names_out(), word_tfidf.idf_))
for w in ["수", "있다", "우도에", "배를"]:
    print(f"IDF {w}: {idf[w]:.2f}")

print()
print("2번 문서의 단어별 TF-IDF 점수:")
row = doc_vectors[1].toarray()[0]
names = word_tfidf.get_feature_names_out()
for j in row.argsort()[::-1]:
    if row[j] > 0:
        print(f"  {names[j]}: {row[j]:.2f}")

### 1-3. 검색기 만들기

검색은 간단합니다. **질문도 같은 방식으로 벡터로 바꾸고**, 문서마다 코사인 유사도를 재서 높은 순서로 보여 줍니다. 6주차의 코사인 유사도 그대로입니다.

같은 뜻의 질문 두 개를 넣어 봅니다. 하나는 문서와 똑같은 모양(`우도에`, `배를`)이고, 하나는 조사가 다릅니다(`우도`, `배`).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def search(query, vectorizer, vectors, top=3):
    scores = cosine_similarity(vectorizer.transform([query]), vectors)[0]
    ranked = scores.argsort()[::-1][:top]
    print(f"질문: {query}")
    if scores.max() == 0:
        print("  모든 문서가 0점이다. 질문의 단어가 문서에 하나도 없다")
        return
    for i in ranked:
        print(f"  {scores[i]:.2f}  {i + 1:2d}. {docs[i]}")

search("우도에 가는 배를 타려면", word_tfidf, doc_vectors)
print()
search("우도 가는 배", word_tfidf, doc_vectors)

### 1-4. 고치기 1: 글자 조각으로 자르기

두 번째 질문은 점수가 모두 0이었습니다. 띄어쓰기로 자르면 `우도` 와 `우도에` 가 다른 단어이기 때문입니다. 이번에는 단어를 **두세 글자 조각**으로 잘라 셉니다. `우도에` 는 `우도`, `도에` 같은 조각으로 나뉘어 `우도` 와 조각을 나눠 가집니다.

In [ ]:
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
char_vectors = char_tfidf.fit_transform(docs)
print("글자 조각 칸:", char_vectors.shape[1], "개")
print()
search("우도 가는 배", char_tfidf, char_vectors)

### 1-5. 고치기 2: 문장 임베딩 모델

문장 임베딩 모델은 **문장 전체를 벡터 하나로** 바꿉니다. 같은 뜻의 문장쌍을 가깝게 두도록 미리 학습된 공개 다국어 모델(paraphrase-multilingual-MiniLM-L12-v2, Apache-2.0)을 내려받아 씁니다. 처음에는 내려받느라 1분 안팎 걸립니다.

내려받기에 실패하면 이 모델을 쓰는 결과만 건너뛰고 나머지는 그대로 돌아갑니다.

In [ ]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME, device="cpu")
    doc_embeddings = model.encode(docs, normalize_embeddings=True)
    print("문장 임베딩:", doc_embeddings.shape[0], "개 x 칸", doc_embeddings.shape[1], "개")
except Exception as e:
    model = None
    print("모델을 불러오지 못했습니다. 임베딩 결과는 건너뜁니다:", e)

def embed_search(query, top=3):
    if model is None:
        print("(모델 없음)")
        return
    scores = (model.encode([query], normalize_embeddings=True) @ doc_embeddings.T)[0]
    ranked = scores.argsort()[::-1][:top]
    print(f"질문: {query}")
    for i in ranked:
        print(f"  {scores[i]:.2f}  {i + 1:2d}. {docs[i]}")

embed_search("우도 가는 배")

### 1-6. 같은 질문을 다르게 물으면

여행자는 같은 것을 여러 방식으로 묻습니다. 질문 다섯 개를 세 방법으로 검색해, 각 방법이 1위로 고른 문서 번호와 점수를 표로 봅니다.

세 방법 모두 **틀리는 경우가 있습니다.** 어느 방법이 어디서 틀리는지 보세요.

In [ ]:
import pandas as pd

questions = [
    "공항에서 시내까지 어떻게 가나요",
    "제주시로 가는 교통편",
    "비 올 때 갈 만한 곳",
    "아이랑 놀기 좋은 바다",
    "제주 전통 음식",
]

def top1(scores):
    i = int(scores.argmax())
    return f"{i + 1}번 ({scores[i]:.2f})" if scores[i] > 0 else "찾지 못함 (0.00)"

table = []
for q in questions:
    row = {"질문": q,
           "단어 TF-IDF": top1(cosine_similarity(word_tfidf.transform([q]), doc_vectors)[0]),
           "글자 조각 TF-IDF": top1(cosine_similarity(char_tfidf.transform([q]), char_vectors)[0])}
    if model is not None:
        row["문장 임베딩"] = top1((model.encode([q], normalize_embeddings=True) @ doc_embeddings.T)[0])
    table.append(row)
pd.DataFrame(table)

## 2. 한 지점만 바꿔 보기

아래 셀의 `TODO` 로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 바꾸기 전 결과를 먼저 확인해 두면 무엇이 달라졌는지 비교할 수 있습니다.

따옴표 안에 **같은 뜻 다른 표현** 문장쌍을 넣으면, 두 문장의 유사도를 세 방법으로 잽니다. 온라인 학습활동에서 만든 문장쌍 세 개를 하나씩 넣어 보세요. 뜻이 반대인 문장쌍(`좋다` 와 `좋지 않다`)도 넣어 비교해 보세요.

In [ ]:
# TODO: 따옴표 안의 두 문장을 바꿔 보세요
sentence_a = "우도 가는 배는 어디서 타요"
sentence_b = "우도행 배편 타는 곳"

# 아래는 그대로 둡니다
pair = [sentence_a, sentence_b]
w = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b").fit(docs + pair)
c = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit(docs + pair)
print(f"A: {sentence_a}")
print(f"B: {sentence_b}")
print(f"단어 TF-IDF     : {cosine_similarity(w.transform([sentence_a]), w.transform([sentence_b]))[0, 0]:.2f}")
print(f"글자 조각 TF-IDF: {cosine_similarity(c.transform([sentence_a]), c.transform([sentence_b]))[0, 0]:.2f}")
if model is not None:
    ea, eb = model.encode(pair, normalize_embeddings=True)
    print(f"문장 임베딩     : {float(ea @ eb):.2f}")

## 3. 확인 질문

1. 1-3에서 `우도 가는 배` 의 점수가 모두 0이 된 이유를 띄어쓰기 단위와 조사로 설명하세요.
2. 1-6 표에서 문장 임베딩이 틀린 질문 하나를 고르고, 1위로 고른 문서가 왜 비슷하다고 판단되었을지 추측해 보세요.
3. TODO 셀에 같은 뜻 다른 표현 문장쌍 세 개를 넣고 세 점수를 옮기세요. 단어 단위 유사도와 문장 단위 유사도가 가장 크게 갈린 쌍은 무엇인가요?

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.

*(여기에 답을 적으세요)*

## 4. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/intronlp-2026)의 `assignments/week-08/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.